# RSNA knee: TensorRT engines for the efficiency pipeline (build + benchmark on a T4)

Run with **2 x T4, internet on**. Inputs:
- the competition data;
- `theoviel/rsna-knee-code-eff`, the code (`efficiency/kaggle/code`);
- `theoviel/rsna-knee-weights-efficiency`, the weights (`output/dataset_efficiency`);
- `theoviel/rsna-knee-onnx-efficiency`, the fp16 ONNX exports (`output/dataset_efficiency_onnx`).

Steps:
1. Install TensorRT for this image's CUDA (`tensorrt-cu12` / `-cu13`) into `/kaggle/working/trtlib`.
2. Build the fp16 engines: the U-Net and the backbones of the recipe's crop models.
3. Benchmark each network: PyTorch fp16 against TensorRT, with output parity.
4. End to end: `pipeline.py` on two disjoint sets of 300 train studies (cold reads both times), PyTorch only against TensorRT.

Outputs, saved as a dataset for the submission:
- `engines/`: `*.engine`, `benchmark.csv`, `trt_meta.json`.
- `trtlib/`: the exact TensorRT the engines need. Engines only load with the same GPU model and TensorRT version.

In [ ]:
import os, subprocess, sys, time, json, glob, shutil

CODE = "/kaggle/input/datasets/theoviel/rsna-knee-code-eff/"
WEIGHTS = "/kaggle/input/datasets/theoviel/rsna-knee-weights-efficiency/"
ONNX = "/kaggle/input/datasets/theoviel/rsna-knee-onnx-efficiency/"
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection/"
WORK = "/kaggle/working/"
TRTLIB, ENGINES = WORK + "trtlib", WORK + "engines"
NETS = "unet,mr85,pl86,cafax87,mk90,91,jc1"        # recipe.json; add mk87,j28 for recipe_plus.json
N_STUDIES = 300
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch
print("torch", torch.__version__, "cuda", torch.version.cuda)

In [ ]:
t = time.time()
cu = torch.version.cuda.split(".")[0]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--target", TRTLIB, f"tensorrt-cu{cu}"], check=True)
# keep only TensorRT itself (numpy etc. come from the image)
for p in os.listdir(TRTLIB):
    if not p.startswith(("tensorrt", "nvidia", "bin")):
        shutil.rmtree(os.path.join(TRTLIB, p), ignore_errors=True) if os.path.isdir(os.path.join(TRTLIB, p)) else os.remove(os.path.join(TRTLIB, p))
sys.path.insert(0, TRTLIB)
os.environ["PYTHONPATH"] = TRTLIB + os.pathsep + os.environ.get("PYTHONPATH", "")
t1 = time.time()
import tensorrt as trt
print("tensorrt", trt.__version__, f"installed in {t1 - t:.0f}s, imported in {time.time() - t1:.1f}s,",
      round(sum(os.path.getsize(f) for f in glob.glob(TRTLIB + "/**", recursive=True) if os.path.isfile(f)) / 1e9, 2), "GB")

In [ ]:
t = time.time()
subprocess.run([sys.executable, os.path.join(CODE, "efficiency", "trt", "build_benchmark.py"), "--onnx", ONNX,
                "--weights", WEIGHTS, "--out", ENGINES, "--nets", NETS], check=True,
               env=dict(os.environ, KNEE_SRC=os.path.join(CODE, "src"), CUDA_VISIBLE_DEVICES="0"))
print(f"build + benchmark: {time.time() - t:.0f}s")

In [ ]:
import pandas as pd

series = pd.read_csv(DATA_DIR + "train_series.csv")
ids = series.StudyInstanceUID.drop_duplicates().sample(2 * N_STUDIES, random_state=0).tolist()
runs = {}
for k, (name, extra) in enumerate([("pytorch", []), ("tensorrt", ["--engines", ENGINES])]):
    pd.DataFrame({"StudyInstanceUID": ids[k * N_STUDIES:(k + 1) * N_STUDIES]}).to_csv(WORK + f"studies_{name}.csv", index=False)
    t = time.time()
    r = subprocess.run([sys.executable, os.path.join(CODE, "efficiency", "pipeline.py"), "--data_dir", DATA_DIR,
                        "--series_dir", "train_series", "--series_csv", DATA_DIR + "train_series.csv",
                        "--weights", WEIGHTS, "--out_dir", f"/tmp/knee_{name}/", "--submission", WORK + f"sub_{name}.csv",
                        "--gpus", "0,1", "--studies", WORK + f"studies_{name}.csv"] + extra,
                       capture_output=True, text=True)
    runs[name] = time.time() - t
    print(f"== {name}: {runs[name]:.0f}s for {N_STUDIES} studies")
    print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "shard done" in l or "engines" in l or "Error" in l))
print({k: f"{v / N_STUDIES * 650:.0f}s per 650 studies (one T4 of the test set)" for k, v in runs.items()})

In [ ]:
print(pd.read_csv(ENGINES + "/benchmark.csv").to_string(index=False))
print(json.load(open(ENGINES + "/trt_meta.json")))
for d in (ENGINES, TRTLIB):
    print(d, round(sum(os.path.getsize(f) for f in glob.glob(d + "/**", recursive=True) if os.path.isfile(f)) / 1e9, 2), "GB")
# Startup cost of TensorRT in a fresh process (each GPU shard pays it): import + deserialize the recipe's engines
r = subprocess.run([sys.executable, "-c", f"""
import time, os, sys; t = time.time()
sys.path.insert(0, '{TRTLIB}'); sys.path.insert(0, '{CODE}efficiency/trt')
import torch; torch.zeros(1, device='cuda'); t1 = time.time()
from trt_runtime import TRTModule
ms = [TRTModule(os.path.join('{ENGINES}', f)) for f in sorted(os.listdir('{ENGINES}')) if f.endswith('.engine')]
print(f'torch + CUDA {{t1 - t:.1f}}s, tensorrt import + {{len(ms)}} engines {{time.time() - t1:.1f}}s')
"""], capture_output=True, text=True)
print(r.stdout, r.stderr[-500:])